
# Bokeh for Time Series Analysis
<hr style="border: 2px solid black;">


<img src="./images/bokeh.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">

<img src="./images/bokeh_at_ag_glance.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">
**Introduction to Bokeh**
Bokeh is an interactive visualization library for Python that targets modern web browsers for presentation.
Unlike Matplotlib, which is primarily designed for static plots, Bokeh excels at creating
interactive plots and dashboards. It can handle large datasets and streaming data,
making it suitable for real-time applications.

**Key Features of Bokeh:**

* **Interactivity:** Built-in support for zooming, panning, hovering, and other interactive tools.
* **Web-Focused:** Generates HTML and JavaScript, making it easy to embed plots in web pages.
* **High Performance:** Can handle large datasets efficiently.
* **Versatility:** Supports a wide range of plot types (lines, bars, scatter plots, etc.).

<hr style="border: 2px solid black;">


**Documentation:**

For comprehensive documentation, please refer to the official Bokeh website: [https://docs.bokeh.org/en/latest/](https://docs.bokeh.org/en/latest/)


<hr style="border: 2px solid black;">


**Lab Exercise:**

Your task is to recreate the time series analysis lab we previously conducted using Pandas,
Matplotlib, and Seaborn, but this time, utilize the Bokeh library for visualization.
This will involve:

1.  Loading and preprocessing the "AirPassengersDates.csv" dataset.
2.  Creating interactive Bokeh plots for:
    * Time series line plots
    * Bar plots of aggregated data
    * Visualizing mean and standard deviation
    * Outlier detection
    * Resampling (upsampling and downsampling)
    * Lag analysis
    * Autocorrelation

Pay close attention to Bokeh's features for interactivity (tools, hover effects) and
its handling of data sources. Aim to replicate the insights and visualizations
from the previous lab while leveraging Bokeh's strengths.

Good luck!
<hr style="border: 2px solid black;">

In [1]:
# =====================================================
# Section 1 : Import des librairies et chargement des données
# =====================================================
import pandas as pd
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import (
    HoverTool, ColumnDataSource, Span, Legend,
    LegendItem, DatetimeTickFormatter, Label, Range1d, RangeTool
)
from bokeh.layouts import column, gridplot
from bokeh.palettes import Category10, Spectral6
from bokeh.transform import factor_cmap

# Activer la sortie Bokeh dans le notebook
output_notebook()

# Chargement du dataset
passenger_df = pd.read_csv("./datasets/AirPassengersDates.csv")
passenger_df["Date"] = pd.to_datetime(passenger_df["Date"])
passenger_df["Month"] = passenger_df["Date"].dt.month
passenger_df["Day"] = passenger_df["Date"].dt.day
passenger_df["Day_Name"] = passenger_df["Date"].dt.day_name()
passenger_df["Year"] = passenger_df["Date"].dt.year

print(f"Shape : {passenger_df.shape}")
print(f"Colonnes : {passenger_df.columns.tolist()}")
passenger_df.head()

Loading BokehJS ...

Shape : (144, 6)
Colonnes : ['Date', '#Passengers', 'Month', 'Day', 'Day_Name', 'Year']


,Date,#Passengers,Month,Day,Day_Name,Year
0,1949-01-01,112,1,1,Saturday,1949
1,1949-02-01,118,2,1,Tuesday,1949
2,1949-03-01,132,3,1,Tuesday,1949
3,1949-04-01,129,4,1,Friday,1949
4,1949-05-01,121,5,1,Sunday,1949


<hr style="border: 2px solid black;">

## Section 2 : Time Series Line Plot (Bokeh)

**Comparaison avec le Lab 5 :**
Dans le lab 5, on utilisait `plt.plot()` pour tracer la série temporelle. Ici, on utilise `figure()` + `p.line()` de Bokeh. La différence majeure : le graphique est immédiatement interactif (pan, zoom, hover) sans code supplémentaire.

<hr style="border: 2px solid black;">

In [2]:
# =====================================================
# Section 2 : Time Series Line Plot avec HoverTool
# =====================================================
source = ColumnDataSource(data=dict(
    date=passenger_df['Date'],
    passengers=passenger_df['#Passengers'],
    month_name=passenger_df['Date'].dt.strftime('%B %Y')
))

hover = HoverTool(tooltips=[
    ('Date', '@date{%F}'),
    ('Passagers', '@passengers{0,0}'),
    ('Mois', '@month_name')
], formatters={'@date': 'datetime'}, mode='vline')

p = figure(
    title='Air Passengers Over Time (1949–1960)',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Nombre de passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)
p.add_tools(hover)
p.line('date', 'passengers', source=source, line_width=1.5, color='steelblue')
p.scatter('date', 'passengers', source=source, size=5, color='steelblue', alpha=0.5)
p.title.text_font_size = '14pt'
show(p)

**Observation :**
- Le HoverTool affiche la date exacte, le nombre de passagers et le mois au survol. Le `mode='vline'` facilite l'exploration sans avoir à viser chaque point.
- La tendance à la hausse est clairement visible : de ~112 passagers en janvier 1949 à ~432 en décembre 1960.
- La saisonnalité est nette : pics chaque été (juillet-août), creux chaque hiver (novembre-février).
- Par rapport à Matplotlib (lab 5), on peut directement zoomer sur une période sans recréer le graphique.

<hr style="border: 2px solid black;">

## Section 3 : Bar Plot — Total de passagers par mois (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 utilisait `sns.barplot()` qui gère l'agrégation implicitement. Avec Bokeh, on pré-calcule avec pandas puis on passe les données via un `ColumnDataSource` + `p.vbar()`.

<hr style="border: 2px solid black;">

In [ ]:
# =====================================================
# Section 3 : Bar Plot — Total passagers par mois
# =====================================================
passengers_per_month = passenger_df.groupby("Month")["#Passengers"].sum().reset_index()
month_labels = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
                'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
passengers_per_month['Month_Label'] = month_labels

source_bar = ColumnDataSource(data=dict(
    month=passengers_per_month['Month_Label'],
    total=passengers_per_month['#Passengers']
))

# Palette de 12 couleurs distinctes (une par mois)
from bokeh.palettes import Category20
palette_12 = Category20[12]

p = figure(
    title='Total de passagers par mois (1949–1960)',
    x_range=month_labels,
    width=900, height=400,
    x_axis_label='Mois',
    y_axis_label='Total passagers',
    tools=['pan', 'box_zoom', 'reset', 'save']
)

p.vbar(x='month', top='total', source=source_bar, width=0.7,
       color=factor_cmap('month', palette=palette_12, factors=month_labels),
       alpha=0.8)

hover = HoverTool(tooltips=[('Mois', '@month'), ('Total', '@total{0,0}')])
p.add_tools(hover)
p.title.text_font_size = '14pt'
p.xgrid.grid_line_color = None
show(p)

**Observation :**
- Les mois d'été (juillet, août) cumulent le plus de passagers — cohérent avec la saison des vacances.
- Le minimum est en novembre/février. La saisonnalité est symétrique autour de l'été.
- Le hover de Bokeh montre le total exact au survol de chaque barre. En Seaborn (lab 5), il faudrait annoter manuellement.
- La palette Spectral colore chaque mois, rendant la lecture plus intuitive que le bleu uniforme de Seaborn.

<hr style="border: 2px solid black;">

## Section 4 : Moyenne et écart-type (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 utilisait `plt.axhline()` pour les lignes horizontales (moyenne, ±1σ). En Bokeh, on utilise `Span()`, un modèle d'annotation qui crée des lignes flottantes interactives.

<hr style="border: 2px solid black;">

In [4]:
# =====================================================
# Section 4 : Moyenne et écart-type avec annotations Span
# =====================================================
mean_val = passenger_df["#Passengers"].mean()
std_val = passenger_df["#Passengers"].std()

source = ColumnDataSource(data=dict(
    date=passenger_df['Date'],
    passengers=passenger_df['#Passengers']
))

p = figure(
    title=f'Passagers avec Moyenne ({mean_val:.0f}) et Écart-type (±{std_val:.0f})',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Nombre de passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

# Série temporelle
p.line('date', 'passengers', source=source, line_width=1.5, color='steelblue', legend_label='Passagers')

# Ligne moyenne
mean_line = Span(location=mean_val, dimension='width', line_color='red',
                 line_dash='dashed', line_width=2)
p.add_layout(mean_line)

# Lignes ±1 écart-type
upper_line = Span(location=mean_val + std_val, dimension='width',
                  line_color='green', line_dash='dashed', line_width=1.5)
lower_line = Span(location=mean_val - std_val, dimension='width',
                  line_color='green', line_dash='dashed', line_width=1.5)
p.add_layout(upper_line)
p.add_layout(lower_line)

# Annotations texte
p.add_layout(Label(x=passenger_df['Date'].iloc[0], y=mean_val + 5,
                   text=f'Moyenne: {mean_val:.0f}', text_color='red', text_font_size='10pt'))
p.add_layout(Label(x=passenger_df['Date'].iloc[0], y=mean_val + std_val + 5,
                   text=f'+1σ: {mean_val + std_val:.0f}', text_color='green', text_font_size='9pt'))
p.add_layout(Label(x=passenger_df['Date'].iloc[0], y=mean_val - std_val + 5,
                   text=f'−1σ: {mean_val - std_val:.0f}', text_color='green', text_font_size='9pt'))

hover = HoverTool(tooltips=[('Date', '@date{%F}'), ('Passagers', '@passengers{0,0}')],
                  formatters={'@date': 'datetime'})
p.add_tools(hover)
p.legend.location = 'top_left'
p.title.text_font_size = '13pt'
show(p)

**Observation :**
- La série dépasse la bande +1σ à partir de ~1957, ce qui confirme la tendance à la hausse. Si la série était stationnaire, les dépassements seraient symétriques haut/bas.
- Les `Span` de Bokeh servent le même rôle que `plt.axhline()`, mais l'ajout de `Label` pour annoter chaque ligne est un plus pour la lisibilité.
- L'interactivité (zoom sur les premières années vs les dernières) permet de voir que la variance augmente avec le temps — la série est hétéroscédastique.

<hr style="border: 2px solid black;">

## Section 5 : Détection des outliers (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 calculait les z-scores et utilisait `plt.scatter()` pour surligner les outliers en rouge. Ici, on fait la même chose mais avec des `circle()` Bokeh et un HoverTool qui affiche le z-score au survol.

<hr style="border: 2px solid black;">

In [5]:
# =====================================================
# Section 5 : Détection des outliers via z-score
# =====================================================
passenger_df["Z_Score"] = (
    passenger_df["#Passengers"] - passenger_df["#Passengers"].mean()
) / passenger_df["#Passengers"].std()
passenger_df["Abs_Z_Score"] = abs(passenger_df["Z_Score"])

# Séparer les points normaux et les outliers (|z| > 2)
normal = passenger_df[passenger_df["Abs_Z_Score"] <= 2]
outliers = passenger_df[passenger_df["Abs_Z_Score"] > 2]

source_normal = ColumnDataSource(data=dict(
    date=normal['Date'], passengers=normal['#Passengers'],
    zscore=normal['Z_Score'].round(2)
))
source_outliers = ColumnDataSource(data=dict(
    date=outliers['Date'], passengers=outliers['#Passengers'],
    zscore=outliers['Z_Score'].round(2)
))

p = figure(
    title='Détection des outliers (|Z-score| > 2)',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Nombre de passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

p.line(passenger_df['Date'], passenger_df['#Passengers'],
       line_width=1, color='steelblue', alpha=0.6, legend_label='Passagers')
p.scatter('date', 'passengers', source=source_normal,
         size=5, color='steelblue', alpha=0.4, legend_label='Normal')
p.scatter('date', 'passengers', source=source_outliers,
         size=10, color='red', alpha=0.8, legend_label='Outliers (|z|>2)')

hover = HoverTool(tooltips=[
    ('Date', '@date{%F}'),
    ('Passagers', '@passengers{0,0}'),
    ('Z-score', '@zscore')
], formatters={'@date': 'datetime'})
p.add_tools(hover)

p.legend.click_policy = 'hide'
p.legend.location = 'top_left'
p.title.text_font_size = '13pt'
show(p)

print(f"\nOutliers détectés ({len(outliers)} points) :")
print(outliers[['Date', '#Passengers', 'Z_Score']].to_string(index=False))


Outliers détectés (5 points) :
      Date  #Passengers  Z_Score
1959-07-01          548 2.231471
1959-08-01          559 2.323164
1960-06-01          535 2.123108
1960-07-01          622 2.848311
1960-08-01          606 2.714940


**Observation :**
- Les outliers (points rouges) se concentrent tous dans les dernières années (1959-1960), sur les mois d'été (juillet-août). Ce ne sont pas de "vrais" outliers au sens statistique — c'est la tendance croissante qui les pousse au-delà de 2σ.
- Le z-score est calculé sur l'ensemble de la série, donc les valeurs récentes (plus élevées à cause de la tendance) apparaissent comme des outliers. Pour une vraie détection d'outliers sur série temporelle, il faudrait d'abord retirer la tendance.
- Cliquer sur "Outliers" dans la légende les masque — utile pour vérifier qu'ils ne biaisent pas la perception du graphique.
- **Avantage Bokeh** : le z-score est affiché au survol de chaque point, ce qui est impossible en Matplotlib statique.

<hr style="border: 2px solid black;">

## Section 6 : Resampling — Upsampling et Downsampling (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 utilisait `resample('D').asfreq()` + `interpolate()` pour l'upsampling et `resample('Y').mean()` pour le downsampling. Ici, on reproduit les mêmes opérations pandas mais on visualise avec Bokeh pour bénéficier du zoom et du hover.

<hr style="border: 2px solid black;">

In [6]:
# =====================================================
# Section 6a : Upsampling — mensuel → journalier
# =====================================================
# Préparer le DataFrame avec Date en index
df_indexed = passenger_df.set_index('Date')[['#Passengers']].copy()

# Upsampling journalier + interpolation linéaire
daily = df_indexed.resample('D').asfreq()
daily['#Passengers'] = daily['#Passengers'].interpolate(method='linear')
daily = daily.reset_index()

source_orig = ColumnDataSource(data=dict(
    date=passenger_df['Date'], passengers=passenger_df['#Passengers']
))
source_daily = ColumnDataSource(data=dict(
    date=daily['Date'], passengers=daily['#Passengers']
))

p = figure(
    title='Upsampling : données mensuelles → journalières (interpolation linéaire)',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

p.line('date', 'passengers', source=source_daily, line_width=1,
       color='coral', alpha=0.7, line_dash='dashed', legend_label='Upsampled (journalier)')
p.scatter('date', 'passengers', source=source_orig, size=6,
         color='steelblue', alpha=0.8, legend_label='Original (mensuel)')

hover = HoverTool(tooltips=[('Date', '@date{%F}'), ('Passagers', '@passengers{0.0}')],
                  formatters={'@date': 'datetime'})
p.add_tools(hover)
p.legend.click_policy = 'hide'
p.legend.location = 'top_left'
p.title.text_font_size = '13pt'
show(p)

In [7]:
# =====================================================
# Section 6b : Downsampling — mensuel → annuel
# =====================================================
yearly = df_indexed.resample('YE')['#Passengers'].mean().reset_index()

source_yearly = ColumnDataSource(data=dict(
    date=yearly['Date'],
    avg_passengers=yearly['#Passengers'].round(1),
    year=yearly['Date'].dt.year.astype(str)
))
source_monthly = ColumnDataSource(data=dict(
    date=passenger_df['Date'], passengers=passenger_df['#Passengers']
))

p = figure(
    title='Downsampling : données mensuelles → moyennes annuelles',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

p.line('date', 'passengers', source=source_monthly, line_width=0.8,
       color='steelblue', alpha=0.4, legend_label='Original (mensuel)')
p.line('date', 'avg_passengers', source=source_yearly, line_width=2.5,
       color='red', legend_label='Moyenne annuelle')
p.scatter('date', 'avg_passengers', source=source_yearly, size=8,
         color='red', alpha=0.8)

hover = HoverTool(tooltips=[
    ('Année', '@year'),
    ('Moyenne', '@avg_passengers{0.0} passagers')
])
p.add_tools(hover)
p.legend.click_policy = 'hide'
p.legend.location = 'top_left'
p.title.text_font_size = '13pt'
show(p)

**Observation Upsampling :**
- L'interpolation linéaire crée une ligne droite entre chaque point mensuel. En zoomant (avantage Bokeh), on voit que les valeurs interpolées sont artificiellement régulières entre les mois — c'est une limite de l'interpolation linéaire.
- Les points bleus (données originales) tombent exactement sur la courbe orange, confirmant que l'interpolation ne modifie pas les valeurs existantes.

**Observation Downsampling :**
- La tendance à la hausse est maintenant limpide : ~130 passagers/mois en 1949 → ~400 passagers/mois en 1960.
- La croissance semble exponentielle plutôt que linéaire (la courbe accélère).
- Le hover affiche la moyenne annuelle exacte — en Matplotlib (lab 5), cette info nécessiterait des annotations manuelles.
- Cliquer sur "Original (mensuel)" dans la légende masque les zigzags saisonniers pour ne garder que la tendance pure.

<hr style="border: 2px solid black;">

## Section 7 : Lag Analysis — Shift (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 utilisait `shift(periods=1)` et `shift(periods=1, freq='MS')` puis `plt.plot()`. Ici, on reproduit l'analyse avec Bokeh et on utilise la légende interactive pour comparer visuellement l'original et les versions décalées.

<hr style="border: 2px solid black;">

In [8]:
# =====================================================
# Section 7 : Lag Analysis — shift et tshift
# =====================================================
df_lag = passenger_df[['Date', '#Passengers']].copy()
df_lag.set_index('Date', inplace=True)
df_lag['Shifted'] = df_lag['#Passengers'].shift(periods=1)
df_lag['tShifted'] = df_lag['#Passengers'].shift(periods=1, freq='MS')

source = ColumnDataSource(data=dict(
    date=df_lag.index,
    original=df_lag['#Passengers'],
    shifted=df_lag['Shifted'],
))

source_tshift = ColumnDataSource(data=dict(
    date_tshift=df_lag['tShifted'].index,
    tshifted=df_lag['tShifted'].values,
))

p = figure(
    title='Lag Analysis : shift(1) vs tshift(1 mois)',
    x_axis_type='datetime',
    width=900, height=400,
    x_axis_label='Date',
    y_axis_label='Passagers',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

p.line('date', 'original', source=source, line_width=1.5,
       color='steelblue', legend_label='Original')
p.line('date', 'shifted', source=source, line_width=1.5,
       color='coral', line_dash='dashed', legend_label='Shift (données décalées de 1)')
p.line('date_tshift', 'tshifted', source=source_tshift, line_width=1.5,
       color='green', line_dash='dotted', legend_label='tShift (index décalé de 1 mois)')

hover = HoverTool(tooltips=[
    ('Date', '@date{%F}'),
    ('Original', '@original{0,0}'),
    ('Shifted', '@shifted{0,0}')
], formatters={'@date': 'datetime'})
p.add_tools(hover)

p.legend.click_policy = 'hide'
p.legend.location = 'top_left'
p.title.text_font_size = '13pt'
show(p)

**Observation :**
- `shift(1)` décale les **valeurs** d'une ligne → la courbe orange (dashed) a la même forme que l'originale mais les valeurs correspondent au mois précédent. Utile pour calculer des variations mois-sur-mois.
- `tshift(1, freq='MS')` décale l'**index** → la courbe verte (dotted) a exactement les mêmes valeurs mais décalées d'un mois vers le futur. Utile pour aligner des prévisions.
- En masquant l'original via la légende, on peut isoler chaque décalage pour mieux comprendre son effet.
- **Cas d'usage** : le `shift` est la base du calcul de la différence première (∆yₜ = yₜ − yₜ₋₁), utilisée pour stationnariser une série temporelle.

<hr style="border: 2px solid black;">

## Section 8 : Autocorrélation (Bokeh)

**Comparaison avec le Lab 5 :**
Le lab 5 utilisait `plot_acf()` de statsmodels. Ici, on calcule les coefficients d'autocorrélation manuellement avec pandas et on les trace avec un bar chart Bokeh + hover.

<hr style="border: 2px solid black;">

In [9]:
# =====================================================
# Section 8 : Autocorrelation Function (ACF) — Bokeh
# =====================================================
max_lag = 30
n = len(passenger_df)
series = passenger_df['#Passengers'].values
mean = series.mean()

# Calcul des coefficients d'autocorrélation
acf_values = []
for lag in range(max_lag + 1):
    if lag == 0:
        acf_values.append(1.0)
    else:
        numerator = np.sum((series[lag:] - mean) * (series[:-lag] - mean))
        denominator = np.sum((series - mean) ** 2)
        acf_values.append(numerator / denominator)

lags = list(range(max_lag + 1))

# Intervalle de confiance à 95% (approximation)
conf_bound = 1.96 / np.sqrt(n)

source_acf = ColumnDataSource(data=dict(
    lag=lags,
    acf=[round(v, 4) for v in acf_values]
))

p = figure(
    title='Autocorrelation Function (ACF) — Air Passengers',
    width=900, height=400,
    x_axis_label='Lag',
    y_axis_label='Autocorrélation',
    tools=['pan', 'box_zoom', 'wheel_zoom', 'reset', 'save']
)

# Barres ACF
p.vbar(x='lag', top='acf', source=source_acf, width=0.4, color='steelblue', alpha=0.8)

# Lignes de confiance ±95%
upper = Span(location=conf_bound, dimension='width',
             line_color='red', line_dash='dashed', line_width=1.5)
lower = Span(location=-conf_bound, dimension='width',
             line_color='red', line_dash='dashed', line_width=1.5)
zero = Span(location=0, dimension='width', line_color='black', line_width=0.5)
p.add_layout(upper)
p.add_layout(lower)
p.add_layout(zero)

hover = HoverTool(tooltips=[('Lag', '@lag'), ('ACF', '@acf{0.0000}')])
p.add_tools(hover)

p.title.text_font_size = '13pt'
show(p)

**Observation :**
- L'ACF décroît lentement depuis le lag 0 (=1 par définition) mais reste significativement positif jusqu'au lag 12-13. C'est typique d'une **série avec tendance** : les valeurs proches dans le temps sont similaires.
- On observe un pic secondaire autour du lag 12, confirmant la **saisonnalité annuelle** (12 mois).
- Tous les lags sont au-dessus de l'intervalle de confiance (lignes rouges), ce qui indique une forte dépendance temporelle — la série n'est pas du bruit blanc.
- Le hover de Bokeh permet de lire la valeur exacte d'ACF pour chaque lag. En Matplotlib/statsmodels (lab 5), les valeurs ne sont pas directement accessibles sans code supplémentaire.

<hr style="border: 2px solid black;">

## Synthèse : Bokeh vs Matplotlib/Seaborn pour l'analyse de séries temporelles

| Section | Lab 5 (Matplotlib/Seaborn) | Lab 6 (Bokeh) | Avantage Bokeh |
|---------|---------------------------|---------------|----------------|
| Line plot | `plt.plot()` | `p.line()` + `HoverTool` | Date et valeur exacte au survol |
| Bar plot | `sns.barplot()` | `p.vbar()` + `factor_cmap` | Hover interactif par barre |
| Moyenne/σ | `plt.axhline()` | `Span()` + `Label()` | Annotations + zoom synchronisé |
| Outliers | `plt.scatter()` rouge | `p.scatter()` + légende `hide` | Filtrage interactif + z-score au hover |
| Resampling | `df.plot()` | `p.line()` + légende `hide` | Masquer/afficher original vs resampleé |
| Lag analysis | `df.plot()` | Multi-line + légende interactive | Comparer visuellement les décalages |
| ACF | `plot_acf()` de statsmodels | `p.vbar()` + `Span` confiance | Valeur ACF exacte au hover |

**Verdict :**
Bokeh apporte l'interactivité (hover, zoom, légende cliquable) à chaque étape de l'analyse. C'est particulièrement utile pour les séries temporelles où l'on veut inspecter des périodes spécifiques ou comparer des valeurs exactes. Matplotlib/Seaborn restent plus rapides pour l'exploration initiale et les figures de publication.